# Cyber Security per la sanità tramite Reinforcement Learning

DeepGuard Inc., un'azienda leader nel settore della sicurezza informatica per le industrie sanitarie, si trova a fronteggiare un aumento della complessità e sofisticazione degli attacchi informatici. La necessità di proteggere le informazioni sensibili dei pazienti e garantire la conformità normativa è cruciale per mantenere la fiducia dei clienti e la sicurezza dei dati.

**Benefici del Progetto:**  
L'implementazione di algoritmi avanzati di Reinforcement Learning per simulare e mitigare scenari di attacco e difesa offre vantaggi significativi:

- **Miglioramento della Difesa:** Addestramento di agenti difensivi in scenari simulati per sviluppare strategie di protezione robuste e adattabili.
- **Identificazione delle Vulnerabilità:** Simulazione di attacchi per identificare e risolvere vulnerabilità nei sistemi di rete prima che possano essere sfruttate da attaccanti reali.
- **Innovazione Tecnologica:** Utilizzo di tecniche avanzate di Reinforcement Learning per promuovere l'innovazione e migliorare continuamente le capacità di difesa informatica.
- **Ottimizzazione delle Risorse:** Automatizzazione della simulazione di attacchi e difese per ridurre il carico di lavoro umano e ottimizzare l'allocazione delle risorse aziendali.

**Dettagli del Progetto:**  
GreenGuard Solutions ha incaricato lo sviluppo di una soluzione avanzata per la sicurezza informatica basata su algoritmi di Reinforcement Learning. Il progetto si focalizza sull'applicazione di due principali algoritmi all'interno dell'ambiente gym-idsgame, specializzato in simulazioni di attacco e difesa in reti informatiche.

**Obiettivi del Progetto:**

- **Algoritmo SARSA:** Utilizzare SARSA per affrontare scenari di "random attack" nell'ambiente gym-idsgame.
- **Algoritmo DDQN:** Implementare Double Deep Q-Network (DDQN) con PyTorch per risolvere scenari di "random attack" e "maximal attack".

**Deliverable:**  
Il progetto richiede la consegna di un notebook Google Colab suddiviso in due sezioni principali, ciascuna dedicata a uno dei due algoritmi di Reinforcement Learning. Ogni sezione dovrà contenere spiegazioni dettagliate delle soluzioni proposte, motivando le scelte effettuate e analizzando i risultati ottenuti.

**Motivazione del Progetto:**  
GreenGuard Solutions pone la massima priorità sulla sicurezza informatica nel settore sanitario. L'utilizzo di algoritmi avanzati di Reinforcement Learning per simulare e mitigare rischi cibernetici consente all'azienda di rafforzare le sue capacità difensive e proteggere le reti di computer contro attacchi sempre più sofisticati. Automatizzando la simulazione di attacchi, GreenGuard ottimizza l'efficienza operativa e si conferma come leader nell'innovazione tecnologica per la sicurezza informatica nel settore sanitario.

# Implementazione

## Import delle dipendenze

In [ ]:
# RLHelper download and installation

!git clone https://github.com/crypto-infinity/rlhelper

In [ ]:
# GymGames download and installation

!git clone https://github.com/Limmen/gym-idsgame
!pip3 install -e ./gym-idsgame

In [29]:
#Dependencies imports

#Generics
import numpy as np
import matplotlib.pyplot as plt
import random
import os

#OAI Gym
import gymnasium as gym
from gym.wrappers.record_video import RecordVideo

#RLHelper
from rlhelper.rlhelper import RLHelper

#Torch
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

#Gym Environment
%cd gym-idsgame/
from gym_idsgame.envs import IdsGameRandomAttackV21Env
from gym_idsgame.envs import IdsGameMaximalAttackV21Env
from gym_idsgame.envs.dao.game_config import GameConfig
from gym_idsgame.envs.dao.idsgame_config import IdsGameConfig
from gym_idsgame.agents.bot_agents.random_attack_bot_agent import RandomAttackBotAgent
from gym_idsgame.agents.bot_agents.attack_maximal_value_bot_agent import AttackMaximalValueBotAgent

[WinError 2] Impossibile trovare il file specificato: 'gym-idsgame/'
c:\Repositories\proai\course8-rl\project\gym-idsgame


In [ ]:
SEED = 52

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
gym.utils.seeding.np_random(SEED)

In [ ]:
# Hyperparameters

GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.1
EPSILON_DECAY = 1000
LEARNING_RATE = 0.0003
BATCH_SIZE = 64
TARGET_UPDATE = 1000
MEMORY_SIZE = 100000
NUM_EPISODES = 300

In [ ]:
VIDEO_PATH = "./videos"

if not os.path.exists(VIDEO_PATH):
    os.makedirs(VIDEO_PATH)

In [ ]:
IMAGES_PATH = "./images"

if not os.path.exists(IMAGES_PATH):
    os.makedirs(IMAGES_PATH)

## Algorithms

In [30]:
class DQN(nn.Module):
    """
    Deep Q-Network model definition.

    Args:
        input_shape (int): Dimension of the input state.
        num_actions (int): Number of possible actions.
    """

    def __init__(self, input_shape, num_actions):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_shape, 512)
        self.fc2 = nn.Linear(512, num_actions)

    def forward(self, x):
        """
        Forward pass of the network.

        Args:
            x (torch.Tensor): Input tensor.
        Returns:
            torch.Tensor: Output Q-values for each action.
        """
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return x

In [33]:
class ReplayBuffer:
    """
    Replay buffer for storing transitions during DDQN training.
    """

    def __init__(self, max_size=100000):
        """
        Initialize the replay buffer.
        Args:
            max_size (int): Maximum number of transitions to store.
        """
        self.memory = deque(maxlen=max_size)

    def push(self, *args):
        """
        Add a transition to the memory.
        Args:
            *args: Transition tuple (state, action, reward, next_state, done).
        """
        self.memory.append(args)

    def sample(self, batch_size):
        """
        Randomly sample a batch of transitions from memory.
        Args:
            batch_size (int): Number of samples to return.
        Returns:
            list: Batch of transitions as arrays.
        """
        batch = random.sample(self.memory, batch_size)
        return [np.array(x) for x in zip(*batch)]

    def __len__(self):
        """
        Return the number of transitions in memory.
        Returns:
            int: Number of transitions stored.
        """
        return len(self.memory)

In [32]:
def _select_action(state, epsilon, n_actions, online_net, device):
        """
        Epsilon-greedy action selection for DDQN.

        Args:
            state (np.ndarray): Current state.
            epsilon (float): Exploration rate.
            n_actions (int): Number of possible actions.
            online_net (nn.Module): Online Q-network.
            device (torch.device): Device to use.
        Returns:
            int: Selected action.
        """
        if random.random() > epsilon:
            with torch.no_grad():
                # EXPLOITATION
                state = torch.FloatTensor(state).unsqueeze(0).to(device)
                return online_net(state).argmax(1).item()
        else:
            # EXPLORATION
            return random.randrange(n_actions)

def _optimize_model(online_dqn, target_dqn, memory, optimizer, batch_size, gamma, device):
    """
    Perform a single optimization step for the online network using DDQN logic.

    Args:
        online_dqn (nn.Module): Online Q-network.
        target_dqn (nn.Module): Target Q-network.
        memory (ReplayBuffer): Experience replay buffer.
        optimizer (torch.optim.Optimizer): Optimizer for the online network.
        batch_size (int): Batch size for optimization.
        gamma (float): Discount factor.
        device (torch.device): Device to use.
    """
    if len(memory) < batch_size:
        return

    state, action, reward, next_state, done = memory.sample(batch_size)

    state = torch.FloatTensor(state).to(device)
    action = torch.LongTensor(action).to(device)
    reward = torch.FloatTensor(reward).to(device)
    next_state = torch.FloatTensor(next_state).to(device)
    done = torch.FloatTensor(done).to(device)

    q_value = online_dqn(state).gather(1, action.unsqueeze(1)).squeeze(1)
    next_actions = online_dqn(next_state).argmax(1).unsqueeze(1)
    next_q_value = target_dqn(next_state).gather(1, next_actions).squeeze(1)
    expected_q_value = reward + gamma * next_q_value * (1 - done)

    loss = (q_value - expected_q_value.detach()).pow(2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [39]:
def ddqn(env, dqn_class, episodes=300, batch_size=64, gamma=0.99, epsilon_start=1.0, epsilon_end=0.1, epsilon_decay=1000, lr=0.0003, target_update=1000, memory_size=100000, max_steps=1000, device=None, verbose=False):
        """
        Double Deep Q-Network (DDQN) algorithm for discrete environments.

        Args:
            env: OpenAI Gym environment.
            dqn_class: Class of the DQN network (must be compatible with PyTorch nn.Module).
            episodes (int): Number of training episodes.
            batch_size (int): Batch size for optimization.
            gamma (float): Discount factor.
            epsilon_start (float): Initial epsilon for exploration.
            epsilon_end (float): Final epsilon value.
            epsilon_decay (int): Number of steps to decay epsilon.
            lr (float): Learning rate.
            target_update (int): Number of episodes between target network updates.
            memory_size (int): Replay buffer size.
            device: PyTorch device.
            max_steps (int): Max steps per episode.
            verbose (bool): If True, print progress.
        Returns:
            list: Rewards per episode.
        """
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        input_shape = env.observation_space.shape[0]
        num_actions = env.action_space.n
        online_net = dqn_class(input_shape, num_actions).to(device)
        target_net = dqn_class(input_shape, num_actions).to(device)
        target_net.load_state_dict(online_net.state_dict())
        optimizer = optim.Adam(online_net.parameters(), lr=lr)
        memory = ReplayBuffer(memory_size)
        epsilon = epsilon_start
        epsilon_decay_step = (epsilon_start - epsilon_end) / epsilon_decay
        episode_rewards = []

        for episode in range(episodes):
            state, _ = env.reset()
            total_reward = 0

            for t in range(max_steps):
                action = RLHelper._select_action(state, epsilon, num_actions, online_net, device)
                next_state, reward, done, _, _ = env.step((0, action))

                defender_reward = reward[1]

                if done and defender_reward <= 0:
                    reward = -1
                total_reward += defender_reward
                memory.push(state, action, reward, next_state, done)
                state = next_state
                RLHelper._optimize_model(online_net, target_net, memory, optimizer, batch_size, gamma, device)
                if done:
                    break
                epsilon = max(epsilon_end, epsilon - epsilon_decay_step)

            if episode % target_update == 0:
                target_net.load_state_dict(online_net.state_dict())
            episode_rewards.append(total_reward)
            if verbose:
                print(f"Episode {episode+1}, Total reward: {total_reward}")

        return episode_rewards

In [ ]:
def sarsa(env, alpha=0.1, gamma=0.99, epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995, episodes=5000, max_steps=100, verbose=False):
    """
    SARSA algorithm for IDSGames (https://github.com/Limmen/gym-idsgame).

    Args:
        env: Gymnasium/OpenAI Gym environment with discrete spaces.
        alpha (float): Learning rate.
        gamma (float): Discount factor.
        epsilon_start (float): Initial exploration rate.
        epsilon_end (float): Final exploration rate.
        epsilon_decay (float): Decay rate for exploration.
        episodes (int): Number of training episodes.
        max_steps (int): Maximum number of steps per episode.
        verbose (bool): If True, print information during learning.
    Returns:
        Q (np.ndarray): Learned Q-table.
        episode_rewards (list): List of total rewards per episode.
    """
    
    n_actions = env.defender_action_space.n
    Q = {}
    episode_rewards = []
    epsilon = epsilon_start

    for ep in range(episodes):
        
        obs = env.reset()[1]
        state = tuple(obs)
        
        done = False
        ep_defender_sum = 0.0

        # Epsilon-greedy policy
        if state not in Q:
            Q[state] = np.zeros(n_actions)

        if np.random.rand() < epsilon:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])

        for step in range(max_steps):

            next_obs, reward, done, info, _ = env.step((0, action))

            defender_reward = reward[1]

            if verbose:
                print(f"Defender Reward: {defender_reward}")

            ep_defender_sum += defender_reward
            next_def_obs = next_obs[1]

            next_state = tuple(next_def_obs.flatten())

            if next_state not in Q:
                Q[next_state] = np.zeros(n_actions)

            if not done:
                #new policy selection
                if np.random.rand() < epsilon:
                    next_action = np.random.randint(n_actions)
                else:
                    next_action = np.argmax(Q[next_state])
            else:
                next_action = None

            current_q = Q[state][action]

            if not done:
                next_q = Q[next_state][next_action]
                td_target = defender_reward + gamma * next_q
            else:
                td_target = defender_reward

            Q[state][action] += alpha * (td_target - current_q)
            state = next_state
            action = next_action

            if done:
                break

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(ep_defender_sum)

        if verbose and (ep+1) % 100 == 0:
            print(f'{ep+1} Episodes Passed')

    return Q, episode_rewards

## Scenario 1 - Random Attack with SARSA

In [3]:
#Environment settings

game_config = GameConfig(dense_rewards=False, dense_rewards_v2=False, dense_rewards_v3=False)

In [4]:
#Agent Setup

attacker_agent = RandomAttackBotAgent(game_config=game_config, env=None)

In [5]:
#Game Config Setup

idsgame_config = IdsGameConfig(game_config=game_config, attacker_agent=attacker_agent)

In [6]:
# Initial setup and reset

env = IdsGameRandomAttackV21Env(idsgame_config=idsgame_config)
obs = env.reset()

In [ ]:
# Training

Q, episode_rewards = sarsa(env)

100 Episodes Passed
200 Episodes Passed
300 Episodes Passed
400 Episodes Passed
500 Episodes Passed
600 Episodes Passed
700 Episodes Passed
800 Episodes Passed
900 Episodes Passed
1000 Episodes Passed
1100 Episodes Passed
1200 Episodes Passed
1300 Episodes Passed
1400 Episodes Passed
1500 Episodes Passed
1600 Episodes Passed
1700 Episodes Passed
1800 Episodes Passed
1900 Episodes Passed
2000 Episodes Passed
2100 Episodes Passed
2200 Episodes Passed
2300 Episodes Passed
2400 Episodes Passed
2500 Episodes Passed
2600 Episodes Passed
2700 Episodes Passed
2800 Episodes Passed
2900 Episodes Passed
3000 Episodes Passed
3100 Episodes Passed
3200 Episodes Passed
3300 Episodes Passed
3400 Episodes Passed
3500 Episodes Passed
3600 Episodes Passed
3700 Episodes Passed
3800 Episodes Passed
3900 Episodes Passed
4000 Episodes Passed
4100 Episodes Passed
4200 Episodes Passed
4300 Episodes Passed
4400 Episodes Passed
4500 Episodes Passed
4600 Episodes Passed
4700 Episodes Passed
4800 Episodes Passed
4

In [40]:
episode_rewards = ddqn(env, DQN)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x11 and 1x512)

In [ ]:
#Plots rewards

## Scenario 2 - Random & Maximal Attack with DDQN